In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-addon-utils
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Usar pós-seleção em cargas de trabalho

<Accordion>
<AccordionItem title="Versões dos pacotes">

O código desta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar essas versões ou versões mais recentes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
qiskit-addon-utils~=0.3.1
```
</AccordionItem>
</Accordion>
Ao otimizar a estratégia de mitigação de erros de uma carga de trabalho, é frequentemente útil filtrar medições que sabidamente foram contaminadas por processos de ruído não-Markovianos (correlacionados). Um método para isso envolve acrescentar ao Circuit uma etapa de pós-processamento que mede qubits ativos e "espectadores" adjacentes, aplica uma rotação lenta a cada qubit e os mede novamente. Nos casos em que as duas medições não confirmam um qubit com flip esperado, o shot é descartado aplicando uma máscara aos resultados.

O pacote [Qiskit addon utilities](https://qiskit.github.io/qiskit-addon-utils/) fornece um conjunto de passes de Transpiler e uma função de pós-seleção para aplicar a máscara. Esta página fornece orientações sobre como incorporar a pós-seleção em suas cargas de trabalho quânticas, usando um estado GHZ de quatro qubits como exemplo.
## Criar carga de trabalho
Comece preparando o circuito a ser executado e transpilado contra um Backend que suporte gates fracionários.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager

circuit = QuantumCircuit(4)
circuit.h(0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.cx(2, 3)
circuit.measure_all()


service = QiskitRuntimeService()
backend = service.least_busy(use_fractional_gates=True)
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

transpiled_circuit = pm.run(circuit)
transpiled_circuit.draw("mpl")

<Image src="../docs/images/guides/post-selection/extracted-outputs/68fa9100-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/post-selection/extracted-outputs/68fa9100-0.svg)

## Adicionar passes de Transpiler de pós-seleção
Em seguida, crie um gerenciador de passes predefinido que inclua os passes [`AddPostSelectionMeasures`](https://qiskit.github.io/qiskit-addon-utils/stubs/qiskit_addon_utils.noise_management.post_selection.transpiler.passes.AddSpectatorMeasures.html) e [`AddSpectatorMeasures`](https://qiskit.github.io/qiskit-addon-utils/stubs/qiskit_addon_utils.noise_management.post_selection.transpiler.passes.AddPostSelectionMeasures.html) do pacote [`qiskit-addon-utils`](https://qiskit.github.io/qiskit-addon-utils/index.html). Isso acrescentará ao Circuit uma sequência de pequenas rotações `RX` angulares (efetivamente produzindo um gate `X` longo) junto com um segundo conjunto de medições.

In [2]:
from qiskit.transpiler import PassManager
from qiskit_addon_utils.noise_management.post_selection import PostSelector
from qiskit_addon_utils.noise_management.post_selection.transpiler.passes import (
    AddPostSelectionMeasures,
    AddSpectatorMeasures,
)


post_selection_pm = PassManager(
    [
        AddSpectatorMeasures(backend.coupling_map, add_barrier=True),
        AddPostSelectionMeasures(x_pulse_type="rx"),
    ]
)

template_circuit_ps = post_selection_pm.run(transpiled_circuit)
template_circuit_ps.draw("mpl", fold=-1, idle_wires=False)

<Image src="../docs/images/guides/post-selection/extracted-outputs/faf50950-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/post-selection/extracted-outputs/faf50950-0.svg)

## Executar programa quântico
Em seguida, prepare um objeto `QuantumProgram` contendo o circuito a ser executado.

In [ ]:
from qiskit_ibm_runtime import QuantumProgram, Executor

shots = 4000

program = QuantumProgram(shots=shots)
program.append_circuit_item(template_circuit_ps)

# Initialize the Executor job and run
executor = Executor(backend)
executor_job = executor.run(program)
print(f"Job ID: {executor_job.job_id()}")

Job ID: d82dumugbeec73alm5g0


Now you can interpret the results. The executor result is a dictionary with several keys.

In [4]:
executor_result = executor_job.result()[0]
executor_result.keys()

dict_keys(['meas', 'spec', 'meas_ps', 'spec_ps'])

Agora você pode interpretar os resultados. O resultado do executor é um dicionário com várias chaves.

In [5]:
post_selector = PostSelector.from_circuit(
    circuit=template_circuit_ps, coupling_map=backend.coupling_map
)

mask_node = post_selector.compute_mask(executor_result, strategy="node")
mask_edge = post_selector.compute_mask(executor_result, strategy="edge")

Both the node and the edge strategies often discard different shots. You can choose to select any of them. This notebook takes a bitwise AND, which is a conservative strategy that retains a shot only if it is passed by both node and edge strategies.

In [ ]:
mask = mask_node & mask_edge
print(f"The combined mask: {mask}")
count_retained = 0

for m in mask:
    count_retained += m

print(
    f"Percentage of the shots retained is after post selection "
    f"{100 * count_retained / shots}"
)

The combined mask: [ True  True  True ...  True  True  True]
Percentage of the shots retained is after post selection 75.225


Essas chaves correspondem aos qubits ativos e espectadores antes das instruções `rx` (`meas` e `spec`) e após as instruções `rx` (`meas_ps` e `spec_ps`). Cada uma delas é um array de arrays baseado no número de shots e qubits. Neste caso, o formato é (1000, 4).

## Criar máscara de pós-seleção
A partir dessas medições, você pode criar uma máscara usando a classe [`PostSelector`](https://qiskit.github.io/qiskit-addon-utils/apidocs/qiskit_addon_utils.noise_management.html#qiskit_addon_utils.noise_management.PostSelector) do `qiskit-addon-utils`. Esta máscara é um array booleano onde cada shot é marcado como `True` ou `False` com base em uma de duas estratégias de pós-seleção. A primeira estratégia, `node`, usa informações de qubit para decidir se um shot de medição deve ser descartado &mdash; e a segunda, `edge`, usa informações de conectividade com vizinhos mais próximos para tomar essa decisão.

In [7]:
counts = {}
counts_ps = {}


for idx, measurement in enumerate(executor_result["meas"]):
    bitstring = ""
    for bit in measurement:
        bitstring += str(int(bit))

    if bitstring in counts:
        counts[bitstring] += 1
    else:
        counts[bitstring] = 1

    # Compute count data for postselected shots based on the mask
    if mask[idx]:
        bitstring = ""
        for bit in measurement:
            bitstring += str(int(bit))

        if bitstring in counts_ps:
            counts_ps[bitstring] += 1
        else:
            counts_ps[bitstring] = 1

for key, val in counts.items():
    counts[key] = val / shots


for key, val in counts_ps.items():
    counts_ps[key] = float(val / count_retained)

Tanto a estratégia de nó quanto a de aresta frequentemente descartam shots diferentes. Você pode escolher qualquer uma delas. Este notebook realiza um AND bit a bit, que é uma estratégia conservadora que retém um shot somente se ele for aprovado por ambas as estratégias de nó e aresta.

In [ ]:
import itertools
from qiskit.visualization import plot_histogram

bitstrings = ["".join(i) for i in itertools.product("01", repeat=4)]
counts_ideal = {}
for bitstring in bitstrings:
    counts_ideal[bitstring] = 0.0
counts_ideal["1111"] = 0.5
counts_ideal["0000"] = 0.5


prob_distance = 0.0
prob_distance_ps = 0.0

for bitstring in counts_ideal.keys():
    dist = 0.0
    dist_ps = 0.0
    if bitstring in counts:
        dist = abs(counts[bitstring] - counts_ideal[bitstring])
    if bitstring in counts_ps:
        dist_ps = abs(counts_ps[bitstring] - counts_ideal[bitstring])
    prob_distance += dist
    prob_distance_ps += dist_ps


print(
    f"Distance from ideal distribution before postselection: "
    f"{1-prob_distance*0.5}"
)
print(
    f"Distance from ideal distribution before after-selection: "
    f"{1-prob_distance_ps*0.5}"
)


plot_histogram([counts, counts_ps], legend=["Normal", "Post selected"])

Distance from ideal distribution before postselection: 0.9015
Distance from ideal distribution before after-selection: 0.9416749750747756


<Image src="../docs/images/guides/post-selection/extracted-outputs/b1ba31b9-1.svg" alt="Output of the previous code cell" />

While postselection can significantly improve result quality by filtering out outcome measurements that were affected by non-Markovian noise, it is not a complete solution to error mitigation on its own. Postselection reduces the impact of certain errors by discarding invalid measurement results, but this comes at the cost of increased sampling overhead and does not address all error mechanisms present in near-term quantum hardware. As a result, it is likely insufficient to rely solely on postselection for more complex or deeper circuits. Instead, postselection is most effective when used as part of a broader error mitigation strategy &mdash; complementing techniques such as measurement error mitigation, noise-aware circuit compilation, or probabilistic error cancellation &mdash; to improve the reliability of quantum workloads while balancing accuracy and resource cost.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Understand how to incorporate [noise learning](/docs/guides/noise-learning) into a quantum workload.
  - Read through other available [error mitigation and suppression](/docs/guides/error-mitigation-and-suppression-techniques) techniques.
  - Learn how to use [spacetime codes](/docs/tutorials/ghz-spacetime-codes) for a low-overhead approach to error detection
</Admonition>